# The Diet Problem

Following a tutorial of JuMP, the modeling language for mathematical optimization in Julia, we consider the diet problem. 

In [1]:
using JuMP
import DataFrames
import GLPK

We encountered ``DataFrames`` in lecture 4.  The ``JuMP`` and ``GLPK`` will need to be installed if they are used for the first time.

## 1. Problem Formulation

Suppose we want to choose the quantity of each food to eat from a set of $n$ foods.

For the $i$th food, we have

1. a cost $c_i$ coefficient, and
2. a nutrient profile $a_{j, i}$, for each nutrient, for $j \in \{1,2, \ldots, m \}$.

In a well balanced meal, for the $j$th nutrient, we have 

1. a lower bound $\ell_j$, and
2. an upper bound $u_j$.

We want to determine the quantities $x_1$, $x_2$, $\ldots$, $x_n$ of each food so that

1. we satisfy all lower and upper bounds on the nutritional requirements, and
2. we minimize the cost.

The cost of the meal is obtained by summing up the quantities multiplied by the cost coefficients:

$$
   \sum_{i=1}^n c_i x_i.
$$

For the $j$th nutrient, we multiply the quantities with the nutrient profile and obtain the linear inequality:

$$
    \ell_j \leq \sum_{j=1}^m a_{j,i} x_i \leq u_j, \quad j=1,2,\ldots,m.
$$

The optimization problem can then be formulated as

$$
    \min \sum_{i=1}^n c_i x_i
$$

subject to

$$
    \ell_j \leq \sum_{j=1}^m a_{j,i} x_i \leq u_j, \quad j=1,2,\ldots,m,
$$

and

$$
    x_i \geq 0, \quad i=1,2,\ldots,n.
$$

The objective function and all constraints are linear, so this is a *linear programming problem*.

## 2. Data for the Problem

The data below is copied from the JuMP tutorial.

In [2]:
foods = DataFrames.DataFrame(
    [
        "hamburger" 2.49 410 24 26 730
        "chicken" 2.89 420 32 10 1190
        "hot dog" 1.50 560 20 32 1800
        "fries" 1.89 380 4 19 270
        "macaroni" 2.09 320 12 10 930
        "pizza" 1.99 320 15 12 820
        "salad" 2.49 320 31 12 1230
        "milk" 0.89 100 8 2.5 125
        "ice cream" 1.59 330 8 10 180
    ],
    ["name", "cost", "calories", "protein", "fat", "sodium"],
)

Row,name,cost,calories,protein,fat,sodium
,Any,Any,Any,Any,Any,Any
1,hamburger,2.49,410,24,26,730
2,chicken,2.89,420,32,10,1190
3,hot dog,1.5,560,20,32,1800
4,fries,1.89,380,4,19,270
5,macaroni,2.09,320,12,10,930
6,pizza,1.99,320,15,12,820
7,salad,2.49,320,31,12,1230
8,milk,0.89,100,8,2.5,125
9,ice cream,1.59,330,8,10,180


We have 9 foods in the rows of the table, $n=9$, and 4 nutrients, $m = 4$, in the last four columns of the table. 

Another data provides the lower and the upper bounds for the nutritional requirements.

In [3]:
limits = DataFrames.DataFrame(
    [
        "calories" 1800 2200
        "protein" 91 Inf
        "fat" 0 65
        "sodium" 0 1779
    ],
    ["name", "min", "max"],
)

Row,name,min,max
,Any,Any,Any
1,calories,1800,2200
2,protein,91,Inf
3,fat,0,65
4,sodium,0,1779


## 3. JuMP formulation

For a linear program, we use ``GLPK``.

In [4]:
model = Model(GLPK.Optimizer)

A JuMP Model
├ solver: GLPK
├ objective_sense: FEASIBILITY_SENSE
├ num_variables: 0
├ num_constraints: 0
└ Names registered in the model: none

We define the variables, from the data in ``foods``.

In [5]:
@variable(model, x[foods.name] >= 0)

1-dimensional DenseAxisArray{VariableRef,1,...} with index sets:
    Dimension 1, Any["hamburger", "chicken", "hot dog", "fries", "macaroni", "pizza", "salad", "milk", "ice cream"]
And data, a 9-element Vector{VariableRef}:
 x[hamburger]
 x[chicken]
 x[hot dog]
 x[fries]
 x[macaroni]
 x[pizza]
 x[salad]
 x[milk]
 x[ice cream]

The objective is to minimize the cost.

In [6]:
@objective(
    model,
    Min,
    sum(food["cost"] * x[food["name"]] for food in eachrow(foods)),
)

2.49 x[hamburger] + 2.89 x[chicken] + 1.5 x[hot dog] + 1.89 x[fries] + 2.09 x[macaroni] + 1.99 x[pizza] + 2.49 x[salad] + 0.89 x[milk] + 1.59 x[ice cream]

The formulation of the constraints uses the ``foods`` and the ``limits``. 

In [7]:
for limit in eachrow(limits)
    intake = @expression(
        model,
        sum(food[limit["name"]] * x[food["name"]] for food in eachrow(foods)),
    )
    @constraint(model, limit.min <= intake <= limit.max)
end

Let us look at the model.

In [8]:
print(model)

## 4. Solving the Linear Programming Problem

In [9]:
optimize!(model)
solution_summary(model)

solution_summary(; result = 1, verbose = false)
├ solver_name          : GLPK
├ Termination
│ ├ termination_status : OPTIMAL
│ ├ result_count       : 1
│ ├ raw_status         : Solution is optimal
│ └ objective_bound    : -Inf
├ Solution (result = 1)
│ ├ primal_status        : FEASIBLE_POINT
│ ├ dual_status          : FEASIBLE_POINT
│ ├ objective_value      : 1.18289e+01
│ └ dual_objective_value : 1.18289e+01
└ Work counters
  └ solve_time (sec)   : 0.00000e+00

Ok, the problem is solved.  Let us now see what the solution looks like.

In [10]:
for food in foods.name
    println(food, " = ", value(x[food]))
end

hamburger = 0.6045138888888888
chicken = 0.0
hot dog = 0.0
fries = 0.0
macaroni = 0.0
pizza = 0.0
salad = 0.0
milk = 6.9701388888888935
ice cream = 2.591319444444441


Interesting solution, we get to drink a lot of milk and eat ice cream.

## 5. Modification of the Problem

Suppose we want to eat one unit of salad, so we add this constraint.

In [11]:
@constraint(model, x["salad"] >= 1)

x[salad] >= 1

In [12]:
optimize!(model)
solution_summary(model)

solution_summary(; result = 1, verbose = false)
├ solver_name          : GLPK
├ Termination
│ ├ termination_status : INFEASIBLE
│ ├ result_count       : 1
│ ├ raw_status         : No feasible primal-dual solution exists.
│ └ objective_bound    : -Inf
├ Solution (result = 1)
│ ├ primal_status        : NO_SOLUTION
│ ├ dual_status          : INFEASIBILITY_CERTIFICATE
│ ├ objective_value      : 1.22686e+01
│ └ dual_objective_value : 1.71303e+00
└ Work counters
  └ solve_time (sec)   : 0.00000e+00

Unfortunately, this requirement is not feasible!